# Nairobi Open-Meteo pipeline

Extract → clean → transform → analyse → model → load → automate. Each stage calls a reusable function.

**Experimental provider-modelled high rainfall; no observed flood outcomes are supplied.** This separate experiment preserves the station pipeline and original data.

[Setup and methodology](scripts/openmeteo_pipeline/README.md) · [Historical Forecast API](https://open-meteo.com/en/docs/historical-forecast-api)

Weather data by Open-Meteo.com / NOAA GFS, CC BY 4.0. The archive stitches early hours from successive model runs. It is not observed station data or a fixed-lead, as-issued forecast archive.

Outputs are cleared for portable publication. See the pipeline README for recorded results.


In [ ]:
from pathlib import Path
import json
import sqlite3
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "scripts/openmeteo_pipeline").is_dir():
    raise RuntimeError("Run this notebook from the repository root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.openmeteo_pipeline.source import Config, extract_data
from scripts.openmeteo_pipeline.runner import analyse, run_pipeline, evaluate_test, generate_schedule
from scripts.nairobi_pipeline.data import clean_data, transform_data, create_targets, split_data
config = Config()
config.validate()

## 1. Extract API data

Annual chunks use the explicit GFS Global model, fixed units, native grid elevation and UTC days. Raw JSON, exact requests, retrieval timestamps and checksums are retained. Nulls are preserved; invalid units/schema/date arrays fail validation. Cached responses are verified before reuse.

Approximate western, southern and eastern Nairobi queries are mapped to returned provider cells. Repeated cell/day records are removed rather than counted as independent station observations. Conflicting measurements fail extraction.

In [ ]:
bundle = extract_data(config)
display(bundle["query_grid_mapping"])
display(bundle["stations"])
display(bundle["raw"].head())
print({key: value for key, value in bundle["manifest"].items() if key != "requests"})
print("Request receipts:", len(bundle["manifest"]["requests"]))

## 2. Clean and audit

Convert numeric fields; check broad physical bounds, temperature ordering, duplicates and calendar coverage. Missing rain remains missing, never zero. There is no interpolation or automatic clipping of extreme rain. The station-specific persistent-value masking policy is not applied to this source; constant runs are audited.

In [ ]:
cleaned, rejected, cleaning_audit = clean_data(bundle["raw"], bundle["stations"])
display(cleaning_audit)
display(rejected.head())
display(cleaned[["station_id", "time", "tavg", "tmin", "tmax", "prcp", "wspd", "pres", "calendar_inserted"]].head())
assert not cleaned.duplicated(["station_id", "time"]).any()
print("Cleaned calendar rows:", len(cleaned))

## 3. Transform into prior-day features

For target date t, use weather through t−1: 1/2-day lags, previous 3/7-day means, previous 7-day rain total, history completeness, season and grid metadata. Missing rolling-window days stay missing. Estimator imputation is fitted later using training data only.

Archive dates do not prove when data were originally available. Prospective lead-time validation requires individual issued forecast runs.

In [ ]:
features = transform_data(cleaned)
display(features.head(10))
print("Feature table shape:", features.shape)

## 4. Analyse quality and training distributions

Coverage and constant-run audits use the full period. Distribution summaries, correlations and monthly rainfall count/mean/maximum use training dates only. Partial months are not described as complete rainfall totals.

In [ ]:
analyses = analyse(cleaned, bundle["stations"], config)
display(analyses["station_quality"])
display(analyses["constant_runs"])
display(analyses["training_descriptive_statistics"])
display(analyses["training_correlations"])
display(analyses["training_monthly_rainfall"].head(12))

## 5. Define target and time partitions

Default target: daily provider rain above the cell's 95th percentile of positive rain through 2022-12-31. This is an experimental high-rainfall definition, not a flood-warning threshold. Missing/unsupported rain has no target.

Training ends 2023; validation is 2024; final testing is 2025. All cells on the same date stay in the same partition. Temporal testing does not establish generalization to new regions.

In [ ]:
model_data, eligibility = create_targets(cleaned, features, config)
snapshot, partitions = split_data(model_data, config)
display(pd.DataFrame(eligibility))
display(pd.DataFrame([
    {"partition": name, "rows": len(part), "first": part.time.min(), "last": part.time.max(), "positive_targets": int(part.target.sum())}
    for name, part in partitions.items()
]))
assert partitions["train"].time.max() < partitions["validation"].time.min()
assert partitions["validation"].time.max() < partitions["test"].time.min()

## 6. Build models, compare, freeze and load

`run_pipeline()` orchestrates the same stages, calls `build_models()`, and loads all tables into a separate SQLite research snapshot. Logistic Regression, Random Forest and XGBoost use chronological training folds and training-only preprocessing. Majority and previous-day rain-persistence baselines accompany the comparison.

Select by validation average precision, with decision threshold fixed at 0.5. Freeze models before final testing. Identical source/config/code/dependencies reuse a verified complete run. This workflow does not modify Supabase or publish app risk predictions.

In [ ]:
run = run_pipeline(config)
selection = json.loads((run / "selection.json").read_text())
display(pd.read_csv(run / "validation_comparison.csv"))
print("Frozen validation-selected model:", selection["selected_model"])
print("CV strategy:", selection["cv_strategy"])
print("Beats best baseline validation AP:", selection["beats_best_baseline_validation_ap"])
print("Run:", run)
with sqlite3.connect(run / "research.sqlite") as connection:
    display(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", connection))
    print("SQLite integrity:", connection.execute("PRAGMA integrity_check").fetchone()[0])

## 7. Explicit frozen final test

Leave the switch off for routine replays. Existing test results may be read below; they are not used for retuning. Once evaluated, this period is no longer an unseen development target.

These metrics describe the provider-derived rainfall target, not flood prediction or independently observed rainfall. Different periods, spatial units and source-defined targets prevent a direct score comparison from establishing improvement over the station experiment.

In [ ]:
EVALUATE_FINAL_TEST = False
if EVALUATE_FINAL_TEST:
    print(evaluate_test(run))
if (run / "test_metrics.csv").exists():
    display(pd.read_csv(run / "test_metrics.csv"))
    print(json.loads((run / "test_evaluation.json").read_text()))
else:
    print("Final test not evaluated; use the explicit CLI command after freezing selection.")

## 8. Automate and prepare the next integration

The CLI automates this entire research workflow. The generated macOS schedule template below is not installed or enabled. It replays a fixed historical study and normally uses cached results; it is not a live daily prediction service.

A later live adapter should retrieve current GFS inputs with the same fields, UTC days and grid handling, record issuance/cutoff/freshness, apply the frozen model and store dated experimental outputs. Do not retrain on every daily refresh. Validate as-issued forecast lead-time skill and actual flood outcomes before presenting operational flood warnings.

In [ ]:
template = generate_schedule(ROOT / ".cache/openmeteo-research.plist", sys.executable)
print("Generated template (not installed/enabled):", template)
print("Research replay: python -m scripts.openmeteo_pipeline run")
print("Final testing: python -m scripts.openmeteo_pipeline evaluate-test", run)